# Итоговый проект по модулю. Прогноз ухода пользователя

Проект рассчитан на **четыре занятия по 90 минут**. Вы пройдёте полный путь от бизнес-вопроса до финальной модели и защиты решения.

**Бизнес-вопрос:** каким пользователям образовательного сервиса стоит заранее предложить помощь, чтобы снизить риск ухода?

**ML-задача:** по данным о пользователе оценить вероятность того, что он уйдёт в ближайший месяц. В обучающих данных факт ухода хранится в столбце `churn`.

**Метрика StackMoreLayers:** ROC-AUC по столбцу `churn_probability`; больше — лучше. Для этой метрики нужно сдавать вероятности, а не метки 0 и 1.


## Маршрут проекта

| Занятие | Главный вопрос | Результат занятия |
|---:|---|---|
| 1 | Как перевести бизнес-задачу в ML-задачу? | Постановка, анализ данных, честное разбиение, простой baseline и первый сабмит |
| 2 | Как подготовить данные без утечки? | Воспроизводимый pipeline, новые признаки и улучшенный сабмит |
| 3 | Какая модель работает лучше и почему? | Честное сравнение моделей, подбор параметров, анализ ошибок и новый сабмит |
| 4 | Как обосновать итоговое решение? | Финальная модель, финальный сабмит, отчёт и защита |

Рабочий код и таблицу экспериментов сохраняйте в этом ноутбуке. После каждого занятия должен оставаться воспроизводимый результат: ноутбук запускается сверху вниз и создаёт корректный CSV-файл.


## Работа со StackMoreLayers

1. Скачайте на странице соревнования `train.csv`, `test.csv` и `sample_submission.csv`.
2. Обучайте модель только по `train.csv`. Ответы для `test.csv` участникам недоступны.
3. Создайте CSV с теми же `id`, что в `test.csv`, и вероятностями в столбце `churn_probability`.
4. Загрузите файл в StackMoreLayers. Система автоматически посчитает ROC-AUC и обновит таблицу лидеров.
5. Записывайте не только результат на платформе, но и результат честной локальной проверки. Таблица лидеров не заменяет validation.

Не используйте `data/private_target.csv`: это служебный файл авторов курса, который не входит в материалы участника.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
DATA_DIR = Path("data")


In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print(train.shape, test.shape)
display(train.head())
display(sample_submission.head())


## Словарь данных

Одна строка — один пользователь образовательного сервиса.

| Группа | Столбцы | Смысл |
|---|---|---|
| Идентификатор | `id` | Номер пользователя; не является содержательным признаком |
| Активность | `account_age_days`, `sessions_previous_30`, `sessions_last_30`, `minutes_per_session`, `days_since_last_activity`, `avg_gap_days` | Возраст аккаунта и активность за два последних периода |
| Взаимодействие | `support_tickets`, `failed_payments`, `homework_completion`, `previous_courses` | Учёба, обращения и оплаты |
| Тариф | `discount_percent`, `plan_price`, `plan_type` | Цена, скидка и вид тарифа |
| Контекст | `mobile_share`, `device`, `region`, `acquisition_channel`, `favorite_topic` | Устройство, регион и источник пользователя |
| Цель | `churn` | 1 — пользователь ушёл, 0 — остался; есть только в `train.csv` |

Перед моделированием проверьте типы столбцов, пропуски, диапазоны значений и долю класса 1. Смысл каждого созданного признака вы должны уметь объяснить на защите.


# Занятие 1. Постановка ML-задачи

## 1.1. Переведите бизнес-вопрос в ML-постановку

Запишите в markdown-ячейке:

- что является объектом;
- какой столбец является целевой переменной;
- почему это классификация, а не регрессия;
- что означает прогноз модели для бизнеса;
- почему главной метрикой выбрана ROC-AUC.


In [ ]:
# Ваш ответ и при необходимости код


## 1.2. Изучите данные

Проверьте размер таблиц, типы столбцов, пропуски, дубликаты, распределение цели и несколько распределений признаков. Сформулируйте не менее трёх наблюдений, которые могут повлиять на моделирование.


In [ ]:
# Ваш код исследования данных


## 1.3. Подготовьте честную локальную проверку

Отделите `id` и `churn` от признаков. Разделите обучающие данные на обучающую и отложенную части со стратификацией. Зафиксируйте `random_state`, чтобы все модели сравнивались на одних объектах.

Объясните, почему ответы отложенной части нельзя использовать при обучении преобразований и выборе признаков.


In [ ]:
# Ваш код разбиения


## 1.4. Постройте простой baseline и сделайте первый сабмит

Соберите минимальный pipeline, посчитайте ROC-AUC на отложенной части, затем обучите его на всём `train.csv` и создайте `submission_lesson_1.csv`.

Если преподаватель выдал `simple_baseline.ipynb`, разберите каждую его часть и измените хотя бы один осмысленный элемент. Не ограничивайтесь запуском готового кода.

**Результат занятия 1:** постановка задачи, три наблюдения о данных, схема разбиения, локальный ROC-AUC и первая запись в таблице лидеров StackMoreLayers.


In [ ]:
# Ваш код baseline и первого сабмита


# Занятие 2. Подготовка данных и pipeline

## 2.1. Соберите воспроизводимую предобработку

- Для числовых столбцов обработайте пропуски. Масштабирование обязательно для линейной модели и допустимо оставить в общем pipeline.
- Для категориальных столбцов обработайте пропуски и сравните One-Hot Encoding с Target Encoding.
- Все преобразования, которые что-либо узнают из данных, должны обучаться только внутри train-части или внутри каждого шага кросс-валидации.
- Если используете Target Encoding, поместите его внутрь pipeline. Кодирование категорий по всей таблице до разбиения создаёт утечку цели.

Зафиксируйте, какой вариант кодирования с какой моделью дал лучший локальный ROC-AUC.


In [ ]:
# Ваш код предобработки и pipeline


## 2.2. Создайте и отберите признаки

Предложите признаки, которые имеют понятный смысл для задачи ухода. Возможные направления: изменение активности между двумя периодами, активность относительно возраста аккаунта, цена после скидки, длительное отсутствие, обращения в поддержку относительно числа сессий.

Для каждого признака запишите гипотезу, затем сравните модель с ним и без него на одной и той же схеме проверки. Не создавайте признаки с использованием `churn` вне корректного pipeline.

**Результат занятия 2:** единый pipeline, таблица проверенных признаков, улучшенный локальный результат и `submission_lesson_2.csv`, загруженный в StackMoreLayers.


In [ ]:
# Ваш код создания и проверки признаков


# Занятие 3. Обучение и подбор моделей

## 3.1. Сравните семейства моделей

На одинаковой кросс-валидации сравните не менее четырёх вариантов:

- логистическую регрессию;
- одно решающее дерево;
- случайный лес;
- градиентный бустинг, включая `CatBoostClassifier`.

XGBoost и LightGBM также доступны и подходят для дополнительных экспериментов. Для каждой строки таблицы сохраняйте название модели, параметры, средний ROC-AUC, разброс результата и время обучения.


In [ ]:
# Ваш код сравнения моделей с cross_val_score или cross_validate


## 3.2. Подберите гиперпараметры

Выберите одну перспективную модель и задайте небольшой осмысленный `param_grid`. Выполните `GridSearchCV` с ROC-AUC и той же схемой кросс-валидации. Объясните смысл каждого перебираемого параметра.

Не подбирайте параметры по результату StackMoreLayers: частые попытки подстроиться под таблицу лидеров дают ненадёжную оценку.


In [ ]:
# Ваш код GridSearchCV


## 3.3. Проанализируйте модель

- Найдите объекты отложенной части с самыми большими ошибками по вероятности.
- Посмотрите важность исходных признаков, например с помощью permutation importance.
- Сформулируйте хотя бы две гипотезы: почему модель ошибается и какой следующий эксперимент это проверит.

**Результат занятия 3:** таблица моделей, подобранные параметры, анализ ошибок и важности признаков, `submission_lesson_3.csv` в StackMoreLayers.


In [ ]:
# Ваш код анализа ошибок и важности признаков


# Занятие 4. Финальная модель и защита

## 4.1. Зафиксируйте финальное решение

Выберите модель по заранее определённой локальной схеме проверки. До финального обучения запишите:

- выбранный pipeline и параметры;
- локальный ROC-AUC;
- почему вы доверяете этой оценке;
- какие эксперименты вы отклонили и почему.

После выбора обучите pipeline на всём `train.csv`, получите вероятности для `test.csv` и создайте `submission_final.csv`.


In [ ]:
# Ваш код финального обучения и создания submission_final.csv


## 4.2. Проверьте файл перед загрузкой

Файл должен содержать ровно два столбца:

- `id` — те же идентификаторы и в том же количестве, что в `test.csv`;
- `churn_probability` — числа от 0 до 1 без пропусков.


In [ ]:
submission_path = Path("submission_final.csv")
submission = pd.read_csv(submission_path)

assert submission.columns.tolist() == ["id", "churn_probability"]
assert len(submission) == len(test)
assert submission["id"].is_unique
assert set(submission["id"]) == set(test["id"])
assert submission["churn_probability"].notna().all()
assert submission["churn_probability"].between(0, 1).all()

print("Формат файла корректен; его можно загружать в StackMoreLayers.")


## 4.3. Подготовьте отчёт и защиту

Заполните `project_report_template.md`. На защите за 5–7 минут покажите:

1. бизнес-задачу и ML-постановку;
2. схему проверки без утечки данных;
3. два-три ключевых эксперимента;
4. финальную модель и локальную метрику;
5. итоговый результат StackMoreLayers;
6. за счёт чего решение улучшилось и что вы попробовали бы дальше.

**Результат занятия 4:** финальный сабмит в StackMoreLayers, воспроизводимый ноутбук, заполненный отчёт и защита проекта.

## Журнал экспериментов

Ведите его с первого занятия.

| № | Изменение | Схема проверки | Локальный ROC-AUC | Результат StackMoreLayers | Вывод |
|---:|---|---|---:|---:|---|
| 1 | Baseline |  |  |  |  |
| 2 |  |  |  |  |  |
| 3 |  |  |  |  |  |
| 4 |  |  |  |  |  |
